## 2 序列模型

### 2.1 理论计算题

**给定字符序列**："ababc"

**词汇表**：{a, b, c}

**一阶马尔可夫模型**：$p(x_t|x_{t-1})$，使用拉普拉斯平滑（加1平滑）

**步骤1：统计所有相邻转移**

序列 "ababc" 中相邻字符对（一阶转移）：
- 位置1-2: a → b
- 位置2-3: b → a  
- 位置3-4: a → b
- 位置4-5: b → c

转移计数：
- a → b: 2次
- b → a: 1次
- b → c: 1次
- 其他所有转移: 0次

**步骤2：统计每个前驱字符的出现次数**

- a出现次数：2次（位置1和3）
- b出现次数：2次（位置2和4）
- c出现次数：1次（位置5）

**步骤3：拉普拉斯平滑公式**

$$p(x_t|x_{t-1}) = \frac{\text{count}(x_{t-1} \to x_t) + 1}{\text{count}(x_{t-1}) + |V|}$$

其中 $|V| = 3$（词汇表大小）

**步骤4：计算 $p(a|b)$**

$$p(a|b) = \frac{\text{count}(b \to a) + 1}{\text{count}(b) + 3} = \frac{1 + 1}{2 + 3} = \frac{2}{5} = 0.4$$

**步骤5：计算 $p(c|b)$**

$$p(c|b) = \frac{\text{count}(b \to c) + 1}{\text{count}(b) + 3} = \frac{1 + 1}{2 + 3} = \frac{2}{5} = 0.4$$

**答案**：
- $p(a|b) = \frac{2}{5} = 0.4$
- $p(c|b) = \frac{2}{5} = 0.4$


In [1]:
import re
from collections import Counter, defaultdict

def preprocess_text(text, n):
    """
    预处理文本，构建词汇表，生成特征序列和标签序列
    
    参数：
        text: 输入文本字符串
        n: 滑动窗口大小（特征长度）
    
    返回：
        vocab: 词汇表字典 {词: ID}
        features: 特征列表，每个特征是一个长度为n的词列表
        labels: 标签列表，每个标签是对应的下一个词
    """
    # 1. 转换为小写，去除标点符号（保留字母和空格）
    text = text.lower()
    # 保留字母和空格，去除其他字符
    text = re.sub(r'[^a-z\s]', '', text)
    
    # 2. 按空格分词
    words = text.split()
    
    # 3. 构建词汇表（按出现频率排序，分配整数ID，从0开始）
    word_counts = Counter(words)
    # 按频率降序排序，频率相同则按字母顺序
    sorted_words = sorted(word_counts.items(), key=lambda x: (-x[1], x[0]))
    vocab = {word: idx for idx, (word, _) in enumerate(sorted_words)}
    
    # 4. 用滑动窗口生成长度为n的特征序列和对应的下一个词标签
    features = []
    labels = []
    
    for i in range(len(words) - n):
        # 特征：从i开始的n个词
        feature = words[i:i+n]
        # 标签：第i+n个词（即下一个词）
        label = words[i+n]
        features.append(feature)
        labels.append(label)
    
    return vocab, features, labels


# 测试示例
if __name__ == "__main__":
    text = "The time machine"
    n = 2
    vocab, features, labels = preprocess_text(text, n)
    
    print("词汇表:", vocab)
    print("特征:", features)
    print("标签:", labels)

词汇表: {'machine': 0, 'the': 1, 'time': 2}
特征: [['the', 'time']]
标签: ['machine']


## 3 循环神经网络

### 3.1 理论计算题

**RNN定义**：

$$h_t = W_{hh}h_{t-1} + W_{hx}x_t$$

$$o_t = W_{oh}h_t$$

**损失函数**（平方损失）：

$$L = \frac{1}{2}\sum_{t=1}^{T}(o_t - y_t)^2$$

**推导 $\frac{\partial L}{\partial W_{hh}}$**：

根据链式法则，损失对 $W_{hh}$ 的梯度是所有时间步贡献的总和：

$$\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^{T} \frac{\partial L}{\partial h_t} \frac{\partial h_t}{\partial W_{hh}}$$

其中 $\frac{\partial h_t}{\partial W_{hh}}$ 需要展开递归依赖。由于 $h_t$ 依赖于 $h_{t-1}$，而 $h_{t-1}$ 又依赖于 $W_{hh}$，我们需要展开所有时间步：

$$\frac{\partial h_t}{\partial W_{hh}} = \sum_{k=1}^{t} \left( \prod_{j=k+1}^{t} \frac{\partial h_j}{\partial h_{j-1}} \right) \frac{\partial h_k}{\partial W_{hh}}$$

其中：
- $\frac{\partial h_j}{\partial h_{j-1}} = W_{hh}$（因为 $h_j = W_{hh}h_{j-1} + \text{const}$）
- $\frac{\partial h_k}{\partial W_{hh}} = h_{k-1}^T$

因此：

$$\frac{\partial h_t}{\partial W_{hh}} = \sum_{k=1}^{t} \left( \prod_{j=k+1}^{t} W_{hh} \right) h_{k-1}^T$$

**完整的梯度表达式**：

$$\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^{T} \frac{\partial L}{\partial o_t} \frac{\partial o_t}{\partial h_t} \frac{\partial h_t}{\partial W_{hh}}$$

$$= \sum_{t=1}^{T} (o_t - y_t) W_{oh} \cdot \sum_{k=1}^{t} \left( \prod_{j=k+1}^{t} W_{hh} \right) h_{k-1}^T$$

**梯度消失或爆炸的条件**：

在通过时间反向传播（BPTT）中，梯度包含乘积项 $\prod_{j=k+1}^{t} W_{hh}$。

令 $\sigma_{\max}$ 为 $W_{hh}$ 的最大奇异值，$\rho(W_{hh})$ 为 $W_{hh}$ 的谱半径：

- **梯度爆炸条件**：当 $\sigma_{\max} > 1$（或 $\rho(W_{hh}) > 1$）时，随着 $t-k$ 增大，乘积项呈指数增长，导致梯度爆炸。

- **梯度消失条件**：当 $\sigma_{\max} < 1$（或 $\rho(W_{hh}) < 1$）时，随着 $t-k$ 增大，乘积项呈指数衰减，导致梯度消失。

- **梯度稳定条件**：当 $\sigma_{\max} = 1$（或 $\rho(W_{hh}) = 1$）时，梯度保持稳定。

**结论**：
- 若 $W_{hh}$ 的谱半径 $\rho(W_{hh}) > 1$，梯度倾向于爆炸
- 若 $W_{hh}$ 的谱半径 $\rho(W_{hh}) < 1$，梯度倾向于消失
- 若 $\rho(W_{hh}) = 1$，梯度保持稳定（理想情况）

In [2]:
import numpy as np

def rnn_cell_forward(x_t, h_prev, W_hh, W_xh, b_h):
    """
    RNN单元前向传播
    
    参数：
        x_t: 当前输入，形状 (batch_size, input_size)
        h_prev: 上一隐藏状态，形状 (batch_size, hidden_size)
        W_hh: 隐藏状态权重，形状 (hidden_size, hidden_size)
        W_xh: 输入权重，形状 (input_size, hidden_size)  [注意：原题笔误为W_hh]
        b_h: 偏置，形状 (hidden_size,)
    
    返回：
        h_t: 当前隐藏状态，形状 (batch_size, hidden_size)
        cache: 缓存用于反向传播
    """
    # 计算隐藏状态
    # h_t = tanh(W_xh * x_t + W_hh * h_prev + b_h)
    # 注意：原题中两个W_hh可能是笔误，这里按标准RNN实现
    h_t = np.tanh(np.dot(x_t, W_xh) + np.dot(h_prev, W_hh) + b_h)
    
    cache = (x_t, h_prev, W_hh, W_xh, b_h, h_t)
    return h_t, cache


def rnn_cell_backward(dh_next, cache):
    """
    RNN单元反向传播
    
    参数：
        dh_next: 上游梯度，即损失对h_t的梯度，形状 (batch_size, hidden_size)
        cache: 前向传播缓存 (x_t, h_prev, W_hh, W_xh, b_h, h_t)
    
    返回：
        dx_t: 损失对x_t的梯度，形状 (batch_size, input_size)
        dh_prev: 损失对h_prev的梯度，形状 (batch_size, hidden_size)
        dW_hh: 损失对W_hh的梯度，形状 (hidden_size, hidden_size)
        dW_xh: 损失对W_xh的梯度，形状 (input_size, hidden_size)
        db_h: 损失对b_h的梯度，形状 (hidden_size,)
    """
    x_t, h_prev, W_hh, W_xh, b_h, h_t = cache
    
    # tanh的导数: d(tanh(z))/dz = 1 - tanh(z)^2
    dh = dh_next * (1 - h_t ** 2)  # 形状 (batch_size, hidden_size)
    
    # 计算各梯度
    # 对于 b_h: db_h = sum(dh over batch)
    db_h = np.sum(dh, axis=0)  # 形状 (hidden_size,)
    
    # 对于 W_hh: dW_hh = h_prev^T * dh
    dW_hh = np.dot(h_prev.T, dh)  # 形状 (hidden_size, hidden_size)
    
    # 对于 W_xh: dW_xh = x_t^T * dh
    dW_xh = np.dot(x_t.T, dh)  # 形状 (input_size, hidden_size)
    
    # 对于 h_prev: dh_prev = dh * W_hh^T
    dh_prev = np.dot(dh, W_hh.T)  # 形状 (batch_size, hidden_size)
    
    # 对于 x_t: dx_t = dh * W_xh^T
    dx_t = np.dot(dh, W_xh.T)  # 形状 (batch_size, input_size)
    
    return dx_t, dh_prev, dW_hh, dW_xh, db_h


# 测试代码
if __name__ == "__main__":
    # 设置参数
    batch_size = 2
    input_size = 3
    hidden_size = 4
    
    # 随机初始化
    x_t = np.random.randn(batch_size, input_size)
    h_prev = np.random.randn(batch_size, hidden_size)
    W_hh = np.random.randn(hidden_size, hidden_size)
    W_xh = np.random.randn(input_size, hidden_size)
    b_h = np.random.randn(hidden_size)
    
    # 前向传播
    h_t, cache = rnn_cell_forward(x_t, h_prev, W_hh, W_xh, b_h)
    print("h_t shape:", h_t.shape)
    
    # 反向传播
    dh_next = np.random.randn(batch_size, hidden_size)
    dx_t, dh_prev, dW_hh, dW_xh, db_h = rnn_cell_backward(dh_next, cache)
    
    print("dx_t shape:", dx_t.shape)
    print("dh_prev shape:", dh_prev.shape)
    print("dW_hh shape:", dW_hh.shape)
    print("dW_xh shape:", dW_xh.shape)
    print("db_h shape:", db_h.shape)

h_t shape: (2, 4)
dx_t shape: (2, 3)
dh_prev shape: (2, 4)
dW_hh shape: (4, 4)
dW_xh shape: (3, 4)
db_h shape: (4,)


## 4 高级循环神经网络

### 4.1 理论计算题

**深度双向RNN结构**：
- L 层
- 每层隐藏单元数 H
- 输入维度 D
- 输出维度 O（仅考虑最后输出层）

**参数计算**：

**1. 第一层（前向+后向）**：

每层RNN包含两个方向的参数（前向和后向），每个方向包含：
- 输入到隐藏的权重和偏置
- 隐藏到隐藏的权重和偏置

前向：
- 输入到隐藏权重 $W_{xh}^{(1,f)}$：形状 $(D, H)$，参数数 $D \times H$
- 输入到隐藏偏置 $b_h^{(1,f)}$：形状 $(H,)$，参数数 $H$
- 隐藏到隐藏权重 $W_{hh}^{(1,f)}$：形状 $(H, H)$，参数数 $H \times H$
- 隐藏到隐藏偏置 $b_h^{(1,f)}$：形状 $(H,)$，参数数 $H$

后向：
- 输入到隐藏权重 $W_{xh}^{(1,b)}$：形状 $(D, H)$，参数数 $D \times H$
- 输入到隐藏偏置 $b_h^{(1,b)}$：形状 $(H,)$，参数数 $H$
- 隐藏到隐藏权重 $W_{hh}^{(1,b)}$：形状 $(H, H)$，参数数 $H \times H$
- 隐藏到隐藏偏置 $b_h^{(1,b)}$：形状 $(H,)$，参数数 $H$

**第一层参数**：

$$\text{Param}_{\text{layer1}} = 2 \times [(D \times H + H) + (H \times H + H)] = 2DH + 2H^2 + 4H$$

**2. 第 l 层（$l \geq 2$，前向+后向）**：

对于第 $l$ 层，输入来自上一层两个方向的拼接，输入维度为 $2H$。

前向：
- 输入到隐藏权重 $W_{xh}^{(l,f)}$：形状 $(2H, H)$，参数数 $2H \times H$
- 输入到隐藏偏置 $b_h^{(l,f)}$：形状 $(H,)$，参数数 $H$
- 隐藏到隐藏权重 $W_{hh}^{(l,f)}$：形状 $(H, H)$，参数数 $H \times H$
- 隐藏到隐藏偏置 $b_h^{(l,f)}$：形状 $(H,)$，参数数 $H$

后向：
- 输入到隐藏权重 $W_{xh}^{(l,b)}$：形状 $(2H, H)$，参数数 $2H \times H$
- 输入到隐藏偏置 $b_h^{(l,b)}$：形状 $(H,)$，参数数 $H$
- 隐藏到隐藏权重 $W_{hh}^{(l,b)}$：形状 $(H, H)$，参数数 $H \times H$
- 隐藏到隐藏偏置 $b_h^{(l,b)}$：形状 $(H,)$，参数数 $H$

**第 $l$ 层参数（$l \geq 2$）**：

$$\text{Param}_{\text{layer }l} = 2 \times [(2H \times H + H) + (H \times H + H)]$$

$$= 2 \times [2H^2 + H + H^2 + H] = 2 \times [3H^2 + 2H] = 6H^2 + 4H$$

**3. 输出层**：

最后一层（第 $L$ 层）的两个方向隐藏状态拼接，维度为 $2H$，输出维度为 $O$。

- 输出权重 $W_{ho}$：形状 $(2H, O)$，参数数 $2H \times O$
- 输出偏置 $b_o$：形状 $(O,)$，参数数 $O$

**输出层参数**：

$$\text{Param}_{\text{output}} = 2H \times O + O$$

**总参数**：

$$\text{Total} = \text{Param}_{\text{layer1}} + \sum_{l=2}^{L} \text{Param}_{\text{layer }l} + \text{Param}_{\text{output}}$$

$$= (2DH + 2H^2 + 4H) + (L-1)(6H^2 + 4H) + (2HO + O)$$

**化简**：

$$\text{Total} = 2DH + 2H^2 + 4H + 6(L-1)H^2 + 4(L-1)H + 2HO + O$$

$$= 2DH + [2 + 6(L-1)]H^2 + [4 + 4(L-1)]H + 2HO + O$$

$$= 2DH + (6L - 4)H^2 + 4LH + 2HO + O$$

**答案**：

$$\boxed{\text{Total} = 2DH + (6L - 4)H^2 + 4LH + 2HO + O}$$

In [3]:
import torch
import torch.nn as nn

class BidirectionalRNNEncoder(nn.Module):
    """
    双向RNN编码器
    """
    def __init__(self, input_dim, hidden_dim, num_layers=1, dropout=0.0):
        super(BidirectionalRNNEncoder, self).__init__()
        
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # 使用PyTorch的RNN实现
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=False,  # 使用 (seq_len, batch, input_dim)
            dropout=dropout if num_layers > 1 else 0.0
        )
        
    def forward(self, X):
        """
        前向传播
        
        参数：
            X: 输入序列，形状 (seq_len, batch, input_dim)
        
        返回：
            outputs: 每个时间步的拼接隐藏状态，形状 (seq_len, batch, 2*hidden_dim)
            final_state: 最终时间步的拼接隐藏状态，形状 (2*hidden_dim,)
        """
        # 使用RNN前向传播
        outputs, h_n = self.rnn(X)
        # outputs: (seq_len, batch, 2*hidden_dim)
        # h_n: (num_layers * 2, batch, hidden_dim)
        
        # 获取最终时间步的隐藏状态（前向和后向的拼接）
        # 前向：最后一层，最后一个时间步；后向：最后一层，第一个时间步（反向）
        # h_n 中索引：前向为 (2*num_layers-2)，后向为 (2*num_layers-1)
        forward_last = h_n[-2, :, :]  # (batch, hidden_dim)
        backward_last = h_n[-1, :, :]  # (batch, hidden_dim)
        final_state = torch.cat([forward_last, backward_last], dim=1)  # (batch, 2*hidden_dim)
        
        return outputs, final_state
    
    def get_final_state(self, X):
        """返回最终时间步的拼接隐藏状态作为序列表示"""
        _, final_state = self.forward(X)
        return final_state


# 手动实现版本（不使用torch.nn.RNN）
class BidirectionalRNNEncoderManual(nn.Module):
    """
    手动实现的双向RNN编码器
    """
    def __init__(self, input_dim, hidden_dim, num_layers=1):
        super(BidirectionalRNNEncoderManual, self).__init__()
        
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # 为每一层创建前向和后向RNN的权重
        self.fw_rnns = nn.ModuleList()
        self.bw_rnns = nn.ModuleList()
        
        for layer in range(num_layers):
            if layer == 0:
                # 第一层：输入维度为 input_dim
                fw_rnn = nn.RNNCell(input_dim, hidden_dim)
                bw_rnn = nn.RNNCell(input_dim, hidden_dim)
            else:
                # 后续层：输入维度为 2*hidden_dim（来自上一层的拼接）
                fw_rnn = nn.RNNCell(2 * hidden_dim, hidden_dim)
                bw_rnn = nn.RNNCell(2 * hidden_dim, hidden_dim)
            
            self.fw_rnns.append(fw_rnn)
            self.bw_rnns.append(bw_rnn)
    
    def forward(self, X):
        """
        手动实现的前向传播
        
        参数：
            X: 输入序列，形状 (seq_len, batch, input_dim)
        
        返回：
            outputs: 每个时间步的拼接隐藏状态，形状 (seq_len, batch, 2*hidden_dim)
            final_state: 最终时间步的拼接隐藏状态，形状 (batch, 2*hidden_dim)
        """
        seq_len, batch_size, _ = X.shape
        
        # 存储每层的前向和后向隐藏状态
        fw_hiddens = []  # 每层的前向隐藏状态列表
        bw_hiddens = []  # 每层的后向隐藏状态列表
        
        # 初始化隐藏状态
        fw_h = [torch.zeros(batch_size, self.hidden_dim) for _ in range(self.num_layers)]
        bw_h = [torch.zeros(batch_size, self.hidden_dim) for _ in range(self.num_layers)]
        
        # 前向传播（正向时间步）
        fw_layer_outputs = []
        for t in range(seq_len):
            x_t = X[t]  # (batch, input_dim)
            layer_input = x_t
            
            for layer in range(self.num_layers):
                if layer > 0:
                    # 非第一层，输入为上一层两个方向的拼接
                    layer_input = torch.cat([fw_h[layer-1], bw_h[layer-1]], dim=1)
                
                fw_h[layer] = self.fw_rnns[layer](layer_input, fw_h[layer])
                layer_input = fw_h[layer]
            
            fw_layer_outputs.append(fw_h[-1].clone())
        
        # 后向传播（反向时间步）
        bw_layer_outputs = [None] * seq_len
        for t in range(seq_len - 1, -1, -1):
            x_t = X[t]  # (batch, input_dim)
            layer_input = x_t
            
            for layer in range(self.num_layers):
                if layer > 0:
                    # 非第一层，输入为上一层两个方向的拼接
                    # 注意：后向传播时，bw_h[layer-1] 是反向时间步的隐藏状态
                    layer_input = torch.cat([fw_h[layer-1], bw_h[layer-1]], dim=1)
                
                bw_h[layer] = self.bw_rnns[layer](layer_input, bw_h[layer])
                layer_input = bw_h[layer]
            
            bw_layer_outputs[t] = bw_h[-1].clone()
        
        # 拼接每个时间步的前向和后向隐藏状态
        outputs = []
        for t in range(seq_len):
            combined = torch.cat([fw_layer_outputs[t], bw_layer_outputs[t]], dim=1)
            outputs.append(combined)
        
        outputs = torch.stack(outputs, dim=0)  # (seq_len, batch, 2*hidden_dim)
        
        # 最终状态：最后一个时间步的前向 + 第一个时间步的后向
        forward_last = fw_layer_outputs[-1]  # (batch, hidden_dim)
        backward_last = bw_layer_outputs[0]  # (batch, hidden_dim)
        final_state = torch.cat([forward_last, backward_last], dim=1)  # (batch, 2*hidden_dim)
        
        return outputs, final_state


# 测试代码
if __name__ == "__main__":
    seq_len = 5
    batch_size = 3
    input_dim = 10
    hidden_dim = 8
    num_layers = 2
    
    X = torch.randn(seq_len, batch_size, input_dim)
    
    # 使用PyTorch内置RNN
    encoder = BidirectionalRNNEncoder(input_dim, hidden_dim, num_layers)
    outputs, final_state = encoder(X)
    print("内置RNN - outputs shape:", outputs.shape)
    print("内置RNN - final_state shape:", final_state.shape)
    
    # 使用手动实现
    encoder_manual = BidirectionalRNNEncoderManual(input_dim, hidden_dim, num_layers)
    outputs_manual, final_state_manual = encoder_manual(X)
    print("手动RNN - outputs shape:", outputs_manual.shape)
    print("手动RNN - final_state shape:", final_state_manual.shape)

内置RNN - outputs shape: torch.Size([5, 3, 16])
内置RNN - final_state shape: torch.Size([3, 16])
手动RNN - outputs shape: torch.Size([5, 3, 16])
手动RNN - final_state shape: torch.Size([3, 16])


## 5 嵌入向量

### 5.1 理论计算题

**Skip-gram模型**：给定中心词 $w_c$，预测上下文词 $w_o$

**负采样目标函数**：

对于每个训练样本 $(w_c, w_o)$，负采样损失函数为：

$$\mathcal{L} = -\log \sigma(\mathbf{v}_{w_c}^T \mathbf{u}_{w_o}) - \sum_{k=1}^{K} \mathbb{E}_{w_n \sim P_n(w)} \log \sigma(-\mathbf{v}_{w_c}^T \mathbf{u}_{w_n})$$

其中 $\sigma(x) = \frac{1}{1+e^{-x}}$ 是 sigmoid 函数。

**完整的目标函数**（用对数似然表示）：

$$J = \log \sigma(\mathbf{v}_{w_c}^T \mathbf{u}_{w_o}) + \sum_{k=1}^{K} \log \sigma(-\mathbf{v}_{w_c}^T \mathbf{u}_{w_{n_k}})$$

**最大化**上述目标函数等价于**最小化**负对数似然：

$$J_{\text{neg}} = -\log \sigma(\mathbf{v}_{w_c}^T \mathbf{u}_{w_o}) - \sum_{k=1}^{K} \log \sigma(-\mathbf{v}_{w_c}^T \mathbf{u}_{w_{n_k}})$$

**符号说明**：
- $\mathbf{v}_{w_c} \in \mathbb{R}^d$：中心词 $w_c$ 的输入向量
- $\mathbf{u}_{w_o} \in \mathbb{R}^d$：上下文词 $w_o$ 的输出向量  
- $\mathbf{u}_{w_{n_k}} \in \mathbb{R}^d$：第 $k$ 个负样本词的输出向量
- $K$：负样本数量
- $P_n(w)$：噪声分布（负样本采样分布）

**负样本采样方法**：

从噪声分布 $P_n(w)$ 中采样 $K$ 个负样本词 $w_{n_1}, w_{n_2}, ..., w_{n_K}$。

常用的噪声分布是**一元语法分布**（unigram distribution）的 $3/4$ 次方：

$$P_n(w) = \frac{\text{count}(w)^{3/4}}{\sum_{w'} \text{count}(w')^{3/4}}$$

这种采样方式使得高频词被采样的概率降低，低频词被采样的概率提高，从而获得更好的词向量。

**完整的目标函数（展开形式）**：

$$J_{\text{neg}} = -\log \sigma(\mathbf{v}_{w_c}^T \mathbf{u}_{w_o}) - \sum_{k=1}^{K} \log \sigma(-\mathbf{v}_{w_c}^T \mathbf{u}_{w_{n_k}})$$

其中 $w_{n_k} \sim P_n(w)$，且 $w_{n_k} \neq w_o$（负样本不包括正样本）。

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

def cbow_forward(context_indices, W, W_out):
    """
    CBOW模型前向传播和损失计算（完整softmax）
    
    参数：
        context_indices: 上下文词索引列表，形状 (batch_size, context_size)
        W: 输入权重矩阵，形状 (V, d)
        W_out: 输出权重矩阵，形状 (d, V)
    
    返回：
        loss: 交叉熵损失值（标量）
        hidden: 隐藏层向量，形状 (batch_size, d)
        probs: 输出概率分布，形状 (batch_size, V)
    """
    batch_size, context_size = context_indices.shape
    V, d = W.shape
    
    # 1. 获取上下文词的嵌入向量
    # context_indices: (batch_size, context_size)
    # 对每个样本，取context_size个词的嵌入向量
    context_embeddings = W[context_indices]  # (batch_size, context_size, d)
    
    # 2. 计算平均上下文向量作为隐藏层
    hidden = torch.mean(context_embeddings, dim=1)  # (batch_size, d)
    
    # 3. 计算输出得分
    scores = torch.matmul(hidden, W_out)  # (batch_size, V)
    
    # 4. 计算softmax概率分布
    probs = F.softmax(scores, dim=1)  # (batch_size, V)
    
    return hidden, probs


def cbow_loss(context_indices, target_indices, W, W_out):
    """
    CBOW模型的完整损失计算
    
    参数：
        context_indices: 上下文词索引列表，形状 (batch_size, context_size)
        target_indices: 目标词索引列表，形状 (batch_size,)
        W: 输入权重矩阵，形状 (V, d)
        W_out: 输出权重矩阵，形状 (d, V)
    
    返回：
        loss: 交叉熵损失值（标量）
    """
    batch_size = context_indices.shape[0]
    
    # 前向传播
    _, probs = cbow_forward(context_indices, W, W_out)
    
    # 计算交叉熵损失
    # 对每个样本，取目标词对应的概率的负对数
    loss = 0.0
    for i in range(batch_size):
        target_idx = target_indices[i]
        # 取出第i个样本中目标词的概率
        prob_target = probs[i, target_idx]
        # 累加负对数似然
        loss += -torch.log(prob_target + 1e-10)  # 加小值防止log(0)
    
    loss = loss / batch_size  # 平均损失
    
    return loss


# 使用nn.CrossEntropyLoss的更简洁实现
class CBOWModel(nn.Module):
    """
    CBOW模型（使用PyTorch的nn.Module）
    """
    def __init__(self, vocab_size, embedding_dim):
        super(CBOWModel, self).__init__()
        
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        
        # 输入权重矩阵 (V, d)
        self.W = nn.Embedding(vocab_size, embedding_dim)
        # 输出权重矩阵 (d, V)
        self.W_out = nn.Linear(embedding_dim, vocab_size, bias=False)
        
    def forward(self, context_indices, target_indices=None):
        """
        前向传播
        
        参数：
            context_indices: 上下文词索引，形状 (batch_size, context_size)
            target_indices: 目标词索引（可选），形状 (batch_size,)
        
        返回：
            如果提供target_indices，返回损失值
            否则返回概率分布
        """
        # 获取上下文词嵌入
        context_emb = self.W(context_indices)  # (batch_size, context_size, d)
        
        # 平均得到隐藏层
        hidden = torch.mean(context_emb, dim=1)  # (batch_size, d)
        
        # 计算输出得分
        scores = self.W_out(hidden)  # (batch_size, vocab_size)
        
        if target_indices is not None:
            # 计算交叉熵损失
            loss = F.cross_entropy(scores, target_indices)
            return loss
        else:
            # 返回概率分布
            return F.softmax(scores, dim=1)


# 测试代码
if __name__ == "__main__":
    # 设置参数
    V = 10  # 词汇表大小
    d = 5   # 嵌入维度
    context_size = 3
    batch_size = 4
    
    # 随机生成数据
    context_indices = torch.randint(0, V, (batch_size, context_size))
    target_indices = torch.randint(0, V, (batch_size,))
    
    # 随机初始化权重
    W = torch.randn(V, d) * 0.01
    W_out = torch.randn(d, V) * 0.01
    
    # 测试函数版本
    hidden, probs = cbow_forward(context_indices, W, W_out)
    loss = cbow_loss(context_indices, target_indices, W, W_out)
    
    print("上下文索引:\n", context_indices)
    print("目标索引:", target_indices)
    print("隐藏层形状:", hidden.shape)
    print("概率分布形状:", probs.shape)
    print("损失值:", loss.item())
    
    # 测试模型版本
    model = CBOWModel(V, d)
    loss_model = model(context_indices, target_indices)
    print("模型损失值:", loss_model.item())

上下文索引:
 tensor([[2, 1, 2],
        [6, 8, 8],
        [8, 0, 4],
        [4, 7, 4]])
目标索引: tensor([2, 9, 4, 5])
隐藏层形状: torch.Size([4, 5])
概率分布形状: torch.Size([4, 10])
损失值: 2.302497148513794
模型损失值: 2.3084170818328857


## 6 注意力机制

### 6.1 理论计算题

**给定**：
- $Q \in \mathbb{R}^{2 \times 4}$（2个查询，每个维度4）
- $K \in \mathbb{R}^{3 \times 4}$（3个键，每个维度4）
- $V \in \mathbb{R}^{3 \times 5}$（3个值，每个维度5）
- $d_k = 4$
- $\text{score} = \frac{QK^T}{\sqrt{d_k}}$

**步骤1：计算得分矩阵 $S = QK^T$**

设 $Q = \begin{bmatrix} q_1 \\ q_2 \end{bmatrix}$，$K = \begin{bmatrix} k_1 \\ k_2 \\ k_3 \end{bmatrix}$

$$S = QK^T = \begin{bmatrix} q_1 \cdot k_1 & q_1 \cdot k_2 & q_1 \cdot k_3 \\ q_2 \cdot k_1 & q_2 \cdot k_2 & q_2 \cdot k_3 \end{bmatrix} \in \mathbb{R}^{2 \times 3}$$

**步骤2：缩放得分矩阵**

$$\tilde{S} = \frac{S}{\sqrt{d_k}} = \frac{S}{2} \in \mathbb{R}^{2 \times 3}$$

**步骤3：对每一行应用 softmax**

$$\text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

设 $\alpha_{ij} = \text{softmax}(\tilde{S}_{i})$ 为第 $i$ 行第 $j$ 列的注意力权重：

$$\alpha_{ij} = \frac{\exp(\tilde{S}_{ij})}{\sum_{m=1}^{3} \exp(\tilde{S}_{im})}$$

注意力权重矩阵 $A \in \mathbb{R}^{2 \times 3}$：

$$A = \begin{bmatrix} \alpha_{11} & \alpha_{12} & \alpha_{13} \\ \alpha_{21} & \alpha_{22} & \alpha_{23} \end{bmatrix}$$

**步骤4：加权求和得到输出**

$$O = A \cdot V \in \mathbb{R}^{2 \times 5}$$

$$O_{i} = \sum_{j=1}^{3} \alpha_{ij} \cdot V_j$$

其中 $V_j$ 是 $V$ 的第 $j$ 行（第 $j$ 个值向量）。

---

**具体数值示例**：

设：
$$Q = \begin{bmatrix} 1 & 0 & 1 & 0 \\ 0 & 1 & 0 & 1 \end{bmatrix}, \quad K = \begin{bmatrix} 1 & 1 & 0 & 0 \\ 0 & 1 & 1 & 0 \\ 0 & 0 & 1 & 1 \end{bmatrix}, \quad V = \begin{bmatrix} 1 & 2 & 3 & 4 & 5 \\ 6 & 7 & 8 & 9 & 10 \\ 11 & 12 & 13 & 14 & 15 \end{bmatrix}$$

**计算 $S = QK^T$**：

第1行：$[1\cdot1+0\cdot1+1\cdot0+0\cdot0,\; 1\cdot0+0\cdot1+1\cdot1+0\cdot0,\; 1\cdot0+0\cdot0+1\cdot1+0\cdot1] = [1, 1, 1]$

第2行：$[0\cdot1+1\cdot1+0\cdot0+1\cdot0,\; 0\cdot0+1\cdot1+0\cdot1+1\cdot0,\; 0\cdot0+1\cdot0+0\cdot1+1\cdot1] = [1, 1, 1]$

$$S = \begin{bmatrix} 1 & 1 & 1 \\ 1 & 1 & 1 \end{bmatrix}$$

**缩放**：

$$\tilde{S} = \frac{1}{2} \begin{bmatrix} 1 & 1 & 1 \\ 1 & 1 & 1 \end{bmatrix} = \begin{bmatrix} 0.5 & 0.5 & 0.5 \\ 0.5 & 0.5 & 0.5 \end{bmatrix}$$

**Softmax**：

每行的 softmax 都为 $[\frac{1}{3}, \frac{1}{3}, \frac{1}{3}]$

$$A = \begin{bmatrix} \frac{1}{3} & \frac{1}{3} & \frac{1}{3} \\ \frac{1}{3} & \frac{1}{3} & \frac{1}{3} \end{bmatrix}$$

**输出**：

$O_1 = \frac{1}{3}(V_1 + V_2 + V_3)$

$= \frac{1}{3}[1+6+11,\; 2+7+12,\; 3+8+13,\; 4+9+14,\; 5+10+15]$

$= \frac{1}{3}[18, 21, 24, 27, 30] = [6, 7, 8, 9, 10]$

$O_2 = O_1 = [6, 7, 8, 9, 10]$

$$O = \begin{bmatrix} 6 & 7 & 8 & 9 & 10 \\ 6 & 7 & 8 & 9 & 10 \end{bmatrix}$$

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

def scaled_dot_product_attention(Q, K, V, mask=None, dropout=None):
    """
    缩放点积注意力
    
    参数：
        Q: 查询矩阵，形状 (..., seq_len_q, d_k)
        K: 键矩阵，形状 (..., seq_len_k, d_k)
        V: 值矩阵，形状 (..., seq_len_v, d_v)，其中 seq_len_v = seq_len_k
        mask: 掩码矩阵，形状 (..., seq_len_q, seq_len_k)
        dropout: dropout层
    
    返回：
        output: 注意力输出，形状 (..., seq_len_q, d_v)
        attention_weights: 注意力权重，形状 (..., seq_len_q, seq_len_k)
    """
    d_k = Q.size(-1)
    
    # 计算得分: Q * K^T / sqrt(d_k)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    
    # 应用掩码（如果有）
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    
    # Softmax
    attention_weights = F.softmax(scores, dim=-1)
    
    # Dropout（如果有）
    if dropout is not None:
        attention_weights = dropout(attention_weights)
    
    # 加权求和
    output = torch.matmul(attention_weights, V)
    
    return output, attention_weights


class MultiHeadAttention(nn.Module):
    """
    多头注意力机制
    """
    def __init__(self, d_model, num_heads, dropout=0.0):
        super(MultiHeadAttention, self).__init__()
        
        assert d_model % num_heads == 0, "d_model必须能被num_heads整除"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.d_v = d_model // num_heads
        
        # 线性投影层
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, X, mask=None):
        """
        前向传播
        
        参数：
            X: 输入，形状 (seq_len, batch, d_model)
            mask: 掩码，形状 (batch, seq_len, seq_len) 或 None
        
        返回：
            output: 输出，形状 (seq_len, batch, d_model)
        """
        seq_len, batch_size, _ = X.shape
        
        # 1. 线性投影得到 Q, K, V
        Q = self.W_q(X)  # (seq_len, batch, d_model)
        K = self.W_k(X)  # (seq_len, batch, d_model)
        V = self.W_v(X)  # (seq_len, batch, d_model)
        
        # 2. 重塑为多头形式
        # (seq_len, batch, d_model) -> (seq_len, batch, num_heads, d_k)
        Q = Q.view(seq_len, batch_size, self.num_heads, self.d_k)
        K = K.view(seq_len, batch_size, self.num_heads, self.d_k)
        V = V.view(seq_len, batch_size, self.num_heads, self.d_v)
        
        # (seq_len, batch, num_heads, d_k) -> (batch, num_heads, seq_len, d_k)
        Q = Q.permute(1, 2, 0, 3)
        K = K.permute(1, 2, 0, 3)
        V = V.permute(1, 2, 0, 3)
        
        # 3. 计算缩放点积注意力（每个头独立）
        # 如果mask不为None，需要扩展维度以匹配多头
        if mask is not None:
            # mask: (batch, seq_len, seq_len) -> (batch, 1, seq_len, seq_len)
            mask = mask.unsqueeze(1)
        
        # 对所有头同时计算
        # Q, K, V: (batch, num_heads, seq_len, d_k/d_v)
        attn_output, attn_weights = scaled_dot_product_attention(
            Q, K, V, mask, self.dropout
        )
        # attn_output: (batch, num_heads, seq_len, d_v)
        
        # 4. 拼接所有头的输出
        # (batch, num_heads, seq_len, d_v) -> (batch, seq_len, num_heads, d_v)
        attn_output = attn_output.permute(0, 2, 1, 3)
        # (batch, seq_len, num_heads, d_v) -> (batch, seq_len, num_heads * d_v) = (batch, seq_len, d_model)
        attn_output = attn_output.reshape(batch_size, seq_len, self.d_model)
        
        # 5. 通过最终线性层
        # (batch, seq_len, d_model) -> (seq_len, batch, d_model)
        attn_output = attn_output.permute(1, 0, 2)
        output = self.W_o(attn_output)
        
        return output


# 更简洁的版本（使用PyTorch内置的MultiheadAttention）
class MultiHeadAttentionSimple(nn.Module):
    """
    使用PyTorch内置多头注意力的简化版本
    """
    def __init__(self, d_model, num_heads, dropout=0.0):
        super(MultiHeadAttentionSimple, self).__init__()
        
        self.mha = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=False  # 使用 (seq_len, batch, d_model)
        )
        
    def forward(self, X, mask=None):
        """
        前向传播
        
        参数：
            X: 输入，形状 (seq_len, batch, d_model)
            mask: 掩码，形状 (seq_len, seq_len) 或 None
        
        返回：
            output: 输出，形状 (seq_len, batch, d_model)
        """
        attn_output, attn_weights = self.mha(X, X, X, attn_mask=mask)
        return attn_output


# 测试代码
if __name__ == "__main__":
    # 设置参数
    seq_len = 10
    batch_size = 4
    d_model = 4
    num_heads = 2
    
    # 创建输入
    X = torch.randn(seq_len, batch_size, d_model)
    
    # 测试多头注意力
    mha = MultiHeadAttention(d_model, num_heads, dropout=0.1)
    output = mha(X)
    
    print("输入形状:", X.shape)
    print("输出形状:", output.shape)
    print("输入和输出形状相同:", X.shape == output.shape)
    
    # 测试内置版本
    mha_simple = MultiHeadAttentionSimple(d_model, num_heads)
    output_simple = mha_simple(X)
    print("内置版本输出形状:", output_simple.shape)

输入形状: torch.Size([10, 4, 4])
输出形状: torch.Size([10, 4, 4])
输入和输出形状相同: True
内置版本输出形状: torch.Size([10, 4, 4])
